In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder,OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,RandomizedSearchCV,KFold,cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

Train_Data=pd.read_csv("C:/Users/Lenovo/Downloads/train (1).csv")
Train_Data['Deck']=Train_Data['Cabin'].str[0]
Train_Data['Deck']=Train_Data['Deck'].fillna('Unknown')
Train_Data["FamilySize"] = Train_Data["SibSp"] + Train_Data["Parch"] + 1
Train_Data["IsAlone"] = (Train_Data["FamilySize"] == 1).astype(int)
Train_Data["Title"] = Train_Data["Name"].str.extract(r",\s*([^\.]+)\.")
rare_titles = ["Lady", "Countess", "Capt", "Col", "Don", "Dr",
               "Major", "Rev", "Sir", "Jonkheer", "Dona"]
Train_Data["Title"] = Train_Data["Title"].replace(rare_titles, "Rare")
Train_Data['Sex']=LabelEncoder().fit_transform(Train_Data['Sex'])
Train_Data=Train_Data.drop(columns=['PassengerId','Name','Ticket','Cabin'])
X=Train_Data.drop(columns=['Survived'])
y=Train_Data['Survived']


## Dividing the features according to their Categories
num_cols=['Age','Fare','SibSp','Parch','FamilySize','IsAlone']
cat_cols=['Pclass','Deck','Embarked','Title']

## Building the Numerical Pipeline
num_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('scaler',StandardScaler())
])

## Building the categorical Pipeline
cat_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

## combining both the Pipelines
preprocessor=ColumnTransformer([
    ('num',num_pipeline,num_cols),
    ('cat',cat_pipeline,cat_cols)
])

## Building the model
model=XGBClassifier(
    random_state=42
)
param_dist = {
    'n_estimators':[100,200,300,400],
    'max_depth':[1,3,5],
    'learning_rate': [0.1, 0.2]
}
kfold=KFold(n_splits=7,shuffle=True,random_state=42)
random_search = RandomizedSearchCV(
    model,
    param_dist,
    n_iter=10,
    cv=5
)

## Building Main Pipeline
pipeline=Pipeline([
    ('preprocessor',preprocessor),
    ('model',random_search)
])
## Train-Test-Split
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    pipeline.fit(X_train,y_train)
    y_pred=pipeline.predict(X_test)
    print(y_pred)
    accuracy=accuracy_score(y_test,y_pred)
    print(accuracy)
    print(f"Fold {fold} Accuracy: {accuracy}")
scores=cross_val_score(pipeline,X,y,cv=kfold,scoring='accuracy')
print(scores)
print('Average_accuracy: ',scores.mean())

Prediction_Data=pd.read_csv("C:/Users/Lenovo/Downloads/test (1).csv")
Prediction_Data['Deck']=Prediction_Data['Cabin'].str[0]
Prediction_Data['Deck']=Prediction_Data['Deck'].fillna('Unknown')
Prediction_Data['Sex']=LabelEncoder().fit_transform(Prediction_Data['Sex'])
Prediction_Data["FamilySize"] = Prediction_Data["SibSp"] + Prediction_Data["Parch"] + 1
Prediction_Data["IsAlone"] = (Prediction_Data["FamilySize"] == 1).astype(int)
Prediction_Data["Title"] = Prediction_Data["Name"].str.extract(r",\s*([^\.]+)\.")
Prediction_Data["Title"] = Prediction_Data["Title"].replace(rare_titles, "Rare")
Prediction_Data=Prediction_Data.drop(columns=['Name','Ticket','Cabin'])
Predictions=pipeline.predict(Prediction_Data)

## Extracting the csv file of Test_Data
submission = pd.DataFrame({
    "PassengerId": Prediction_Data["PassengerId"],
    "Survived": Predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)
import os
print(os.getcwd())





[0 0 0 1 0 0 1 1 1 0 1 0 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 1 0 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 1 0 0
 1 1 0 1 0 0 0 1 0 0 1 1 1 1 0 1 1 1 0 1 0 0 1 1 1 0 0 0 1 1 0 0 0 1 0 1 1
 1 1 1 0 1 1 0 0 0 0 0 1 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 1
 1 0 1 1 0 0 1 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0]
0.8435754189944135
Fold 1 Accuracy: 0.8435754189944135
[0 0 0 1 0 0 1 0 0 0 1 1 1 0 0 0 0 0 1 1 1 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 1
 1 0 1 0 0 0 1 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 1 1 0 1 1 1 1 0 1 0 0 1 1 0 1
 1 0 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 0 1 0 0 1 0 1 1 0 0
 0 0 0 0 0 1 1 0 1 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0 1 0 0 1 0 0 0 0 1 0 0 0 1
 1 0 0 0 1 0 0 0 0 0 1 1 0 1 0 1 0 0 1 0 0 1 0 1 0 0 1 0 1 1]
0.8539325842696629
Fold 2 Accuracy: 0.8539325842696629
[1 0 0 0 1 1 0 1 1 0 0 0 0 0 1 1 0 0 0 0 0 1 1 0 1 0 1 1 0 0 1 1 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 1 0 1 1 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0
 0 0 0 0 0 0 0